# Sample: XNAT from a Scout notebook

This notebook demonstrates the Phase C `scout_xnat_auth` helper. It opens an
[xnatpy](https://xnat.readthedocs.io/) connection authenticated as you (the
logged-in JupyterHub user) and pokes a few read-only XNAT endpoints.

Prerequisites:

- `scout_xnat_auth.py` sitting next to this notebook.
- You're a member of the `scout-user` Keycloak group (so XNAT will
  provision your account on first contact).
- The notebook has been spawned recently enough that the cached Keycloak
  access token in `auth_state` is still valid (the helper auto-refreshes
  on 401, so this matters mostly for the very first cell).

## 1. Connect

The first call to `connect()` runs the full bearer pipeline on the XNAT
side: validate JWT → exchange for an `xnat`-audience token → validate
again → provision the local `keycloak-<sub>` user → set a `JSESSIONID`
on the response. Subsequent calls on this `connection` reuse the
cookie.

In [ ]:
import scout_xnat_auth

connection = scout_xnat_auth.connect()
print("Logged in as:", connection.logged_in_user)

## 2. List projects

If this is a freshly-installed XNAT with no data yet, the listing will be
empty — that's fine. The point is that the call returns 200 instead of
401 / 403.

In [ ]:
projects = list(connection.projects.values())
print(f"{len(projects)} project(s) visible to {connection.logged_in_user}")
for project in projects:
    print(" ", project.id, "--", getattr(project, "name", "(no name)"))

## 3. Hit the raw REST API

`xnatpy` exposes `connection.get` / `.post` / `.put` / `.delete` for
anything the ORM doesn't cover. The helper's `JSESSIONID` cookie is
attached transparently by xnatpy's `requests.Session`.

In [ ]:
response = connection.get("/xapi/users/profile")
profile = response.json()
print("username =", profile.get("username"))
print("email    =", profile.get("email"))
print("verified =", profile.get("verified"))

## 4. List subjects across all visible projects (if any)

In [ ]:
for project in projects:
    subjects = list(project.subjects.values())
    print(f"{project.id}: {len(subjects)} subject(s)")
    for subject in subjects[:5]:
        print("   ", subject.label)

## 5. Disconnect cleanly

Optional — xnatpy will tear the connection down on garbage-collection,
but explicit shutdown stops the keepalive thread immediately.

In [ ]:
connection.disconnect()